In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

# Define constants
c = 299792.458  # Speed of light in km/s

# Compute f_DE(z) using the w0-wa parametrization
def f_de(z, w0, wa):
    return (1/(1+z))**(-3*(1+w0+wa))*np.exp(-3*wa*(z/(1+z)))

# Define the Hubble parameter H(z) based on the Friedmann equation
def hubble_parameter(z, omega_m0, omega_r0, hrdrag, w0, wa):
    # Omega_de0 from the flat universe condition
    omega_de0 = 1 - omega_m0 - omega_r0
    # Hubble parameter H(z)
    return np.sqrt(omega_m0 * (1 + z)**3 + omega_r0 * (1 + z)**4 + omega_de0 * f_de(z, w0, wa))

# Calculate D_H(z) / r_d based on Eq. 2.8
def D_H_over_rd(z, hrdrag, omega_m0, omega_r0, w0, wa):
    hz = hubble_parameter(z, omega_m0, omega_r0, hrdrag, w0, wa) * hrdrag 
    return c / hz

# Calculate D_M(z) / r_d based on Eq. 2.9
def D_M_over_rd(z, hrdrag, omega_m0, omega_r0, w0, wa):
    def integrand(z_prime):
        return 1 / hubble_parameter(z_prime, omega_m0, omega_r0, hrdrag, w0, wa)
    
    integral, _ = quad(integrand, 0, z)
    return c * integral / hrdrag

# Calculate D_V(z) based on D_H and D_M
def D_V_over_rd(z, hrdrag, omega_m0, omega_r0, w0, wa):
    DH = D_H_over_rd(z, hrdrag, omega_m0, omega_r0, w0, wa)
    DM = D_M_over_rd(z, hrdrag, omega_m0, omega_r0, w0, wa)
    return (z * DM**2 * DH)**(1/3)

# Define the likelihood function
def likelihood(theta, data, quantity_array, covariance):
    # Unpack parameters
    hrdrag, omega_m, w0, wa = theta
    
    # Get model predictions based on the parameters for each z
    model_prediction = model(hrdrag, omega_m, w0, wa, data, quantity_array)
    
    # Calculate the likelihood (assuming Gaussian)
    diff = data[:, 1] - model_prediction
    return -0.5 * np.dot(diff.T, np.linalg.solve(covariance, diff))

# Define your model function
def model(hrdrag, omega_m, w0, wa, data, quantity_array):
    # Initialize an array to hold model predictions
    predictions = np.zeros(data.shape[0])
    
    omega_r0 = 0  # Radiation density, can be updated as needed
    
    for i in range(data.shape[0]):
        z = data[i, 0]  # Redshift value
        quantity = quantity_array[i]  # Use the quantity from the separate array
        
        # Compute D_H, D_M, D_V based on the quantity
        if quantity == 'DH_over_rs':
            predictions[i] = D_H_over_rd(z, hrdrag, omega_m, omega_r0, w0, wa)
        elif quantity == 'DM_over_rs':
            predictions[i] = D_M_over_rd(z, hrdrag, omega_m, omega_r0, w0, wa)
        elif quantity == 'DV_over_rs':
            predictions[i] = D_V_over_rd(z, hrdrag, omega_m, omega_r0, w0, wa)
        else:
            raise ValueError(f"Unknown quantity: {quantity}")
    
    return predictions
        
# Load data and covariance matrix from files
def load_data_and_covariance(data_path, cov_path):
    # Load data, treating the first two columns as floats and the third as strings
    data = np.loadtxt(data_path, dtype={'names': ('z', 'value', 'quantity'), 'formats': ('f4', 'f4', 'U20')}, comments='#')
    covariance = np.loadtxt(cov_path)
    
    # Create a numerical data array (first two columns)
    data_array = np.zeros((data.shape[0], 2))
    data_array[:, 0] = data['z']
    data_array[:, 1] = data['value']
    
    # Create a separate array for the quantity strings
    quantity_array = data['quantity']
    
    return data_array, quantity_array, covariance

# Paths to the data and covariance matrix
data_path = '/home/domenicogiardino/Documents/cobaya_project/cobaya_env/cobaya/cobaya/data/bao_data/desi_2024_gaussian_bao_ALL_GCcomb_mean.txt'
cov_path = '/home/domenicogiardino/Documents/cobaya_project/cobaya_env/cobaya/cobaya/data/bao_data/desi_2024_gaussian_bao_ALL_GCcomb_cov.txt'

# Load data and covariance matrix
data, quantity_array, covariance = load_data_and_covariance(data_path, cov_path)

# MCMC parameters
initial_theta = np.array([10000, 0.32, -1, 0])  # Initial guess for parameters (H0rd, Omega_m, w0, wa)
target_accepted = 50000  # Number of MCMC accepted iterations
adapt_interval = 1000 # N.iteration after whic print acceptance rate
#proposal_scale = np.diag([400**2, 0.03**2, 0.3**2, 1**2])  # Initial covariance matrix (diagonal)  # Proposal scale for each parameter
proposal_scale = [
    [1.88686746e+05, -1.39251065e+01, -1.04263164e+02,  2.82451863e+02],
    [-1.39251065e+01,  1.92454283e-03,  8.23048745e-03, -3.56666484e-02],
    [-1.04263164e+02,  8.23048745e-03,  6.68132821e-02, -2.06040809e-01],
    [2.82451863e+02, -3.56666484e-02, -2.06040809e-01,  8.93192867e-01]
] 

In [ ]:
# MCMC Sampling function using MPI for parallelization

def mcmc_sampling(data, quantity_array, covariance, initial_theta, proposal_scale, target_accepted, adapt_interval):
    theta = np.random.multivariate_normal(initial_theta, proposal_scale)  # Adjust rescaling
    samples = []  # Initialize as a list
    acceptance_count = 0
    accepted_count = 0
    iteration = 0

    while accepted_count < target_accepted:

        # Propose a new sample
        theta_prime = np.random.multivariate_normal(theta, proposal_scale)
        iteration += 1
        # Apply priors: w0 and wa should be within [-2, 3]
        if not (-3 <= theta_prime[2] <= 1 and -3 <= theta_prime[3] <= 2 and 0.01 <= theta_prime[1] <= 0.98 and theta_prime[2] + theta_prime[3] <= 0):
            samples.append(theta)  # Append the current sample
            continue  # Skip if proposal is out of bounds
        # Calculate likelihoods (local to each process)
        log_likelihood_current = likelihood(theta, data, quantity_array, covariance)
        log_likelihood_proposed = likelihood(theta_prime, data, quantity_array, covariance)

        # Calculate acceptance ratio
        acceptance_ratio = np.exp(log_likelihood_proposed - log_likelihood_current)

        if np.random.rand() < acceptance_ratio:
            theta = theta_prime
            acceptance_count += 1
            accepted_count += 1

        samples.append(theta)  # Append the current sample
        # Adapt the proposal scale (only after a certain number of accepted samples)
        if (accepted_count % adapt_interval == 0) and (acceptance_count > 0):
            acceptance_rate = acceptance_count / iteration
            print(f"Accepted {accepted_count} samples. Current acceptance rate = {acceptance_rate:.3f}")
            acceptance_count = 0
            iteration = 0
            if accepted_count >= 1000:
                # Calculate covariance of the samples for adjusting proposal scale
                variances = np.cov(np.array(samples), rowvar=False)  # Convert to array for covariance calculation
                proposal_scale = variances * (1)  # Example scaling, adjust multiplier as needed
        
            
    return np.array(samples)  # Convert to NumPy array before returning


# Run MCMC sampling with adaptive proposal scaling
samples = mcmc_sampling(data, quantity_array, covariance, initial_theta, proposal_scale, target_accepted, adapt_interval)

# Plotting the results
plt.figure(figsize=(10, 10))

# Plot H0rd samples
plt.subplot(2, 2, 1)
plt.plot(samples[:,0], label='H0rd')
plt.title('Samples for H0rd')
plt.xlabel('Iteration')
plt.ylabel('Value')
plt.legend()

# Plot Omega_m samples
plt.subplot(2, 2, 2)
plt.plot(samples[:, 1], label='Omega_m', color='orange')
plt.title('Samples for Omega_m')
plt.xlabel('Iteration')
plt.ylabel('Value')
plt.legend()

# Plot w0 samples
plt.subplot(2, 2, 3)
plt.plot(samples[:, 2], label='w0', color='green')
plt.title('Samples for w0')
plt.xlabel('Iteration')
plt.ylabel('Value')
plt.legend()

# Plot wa samples
plt.subplot(2, 2, 4)
plt.plot(samples[:, 3], label='wa', color='red')
plt.title('Samples for wa')
plt.xlabel('Iteration')
plt.ylabel('Value')
plt.legend()

plt.tight_layout()
plt.show()


Accepted 1000 samples. Current acceptance rate = 0.177
Accepted 2000 samples. Current acceptance rate = 0.240
Accepted 3000 samples. Current acceptance rate = 0.243
Accepted 4000 samples. Current acceptance rate = 0.237
Accepted 5000 samples. Current acceptance rate = 0.244
Accepted 6000 samples. Current acceptance rate = 0.234
Accepted 7000 samples. Current acceptance rate = 0.237
Accepted 8000 samples. Current acceptance rate = 0.241
Accepted 9000 samples. Current acceptance rate = 0.246
Accepted 10000 samples. Current acceptance rate = 0.241
Accepted 11000 samples. Current acceptance rate = 0.248
Accepted 12000 samples. Current acceptance rate = 0.259
Accepted 13000 samples. Current acceptance rate = 0.255
Accepted 14000 samples. Current acceptance rate = 0.239
Accepted 15000 samples. Current acceptance rate = 0.252
Accepted 16000 samples. Current acceptance rate = 0.237
Accepted 17000 samples. Current acceptance rate = 0.252
Accepted 18000 samples. Current acceptance rate = 0.262
A

In [ ]:
import numpy as np
import getdist.plots as gpl
# Show plots inline, and load main getdist plot module and samples class
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import sys, os
sys.path.insert(0, os.path.realpath(os.path.join(os.getcwd(), '..')))
from getdist import plots, MCSamples
import getdist
import matplotlib.pyplot as plt
import IPython

## Create MCSamples object
param_names = ['H0rd', 'Omega_m', 'w0', 'wa']
g = MCSamples(samples=samples, names=param_names)
## Set up the plot
gplot = gpl.getSubplotPlotter()
gplot.triangle_plot(g, filled=True)

names = ['H0rd', 'Omega_m', 'w0', 'wa']
labels =  ['H0rd', 'Omega_m', 'w0', 'wa']
sample = MCSamples(samples=samples,names = names, labels = labels)
# Create a single plotter for the 2D plot
g2d = plots.get_single_plotter(width_inch=4, ratio=1)
g2d.plot_2d(sample, 'w0', 'wa', filled=False, ls=['--'])
g2d.add_x_marker(-1, lw=1), g2d.add_y_marker(0, lw=1)

In [ ]:
import arviz as az #https://python.arviz.org/en/latest/getting_started/Introduction.html
print("GETDIST INTERVAL")
for i, name in enumerate(names):
    mean=sample.mean(names[i])
    std=sample.std(names[i])
    print("",sample.getInlineLatex(names[i],limit=1),f"std is : {std}")# lim 1 68 ,lim 2 95

print("\nHDR INTERVAL")
for i, name in enumerate(names):   
    mean=sample.mean(names[i])
    hdi = az.hdi(samples[:,i], hdi_prob=0.68)  # 68% HDR
    print(f" {name} : mean = {mean:.4f}, 68% CI = [{(-mean+hdi[0]):.4f}, {(-mean+hdi[1]):.4f}]")